# 02 — Cleaning & Analytical Sample Construction

## tl;dr

The deterministic Phase 2.1 pipeline creates five separate sample layers: 6,602,582 RAW observations; 6,461,297 valid movement observations; 4,707,776 valid-price observations; 4,706,611 core cereal-price observations; and 4,508,667 eligible consecutive price comparisons. Three approved non-food merchandise UPCs are excluded only from CORE and downstream views through a versioned config.

## Context & Methods

The notebook orchestrates `../src/clean_data.py`; business rules do not live only in notebook cells. Full derived data are stored once in SQLite and exposed through named sample views. Compact CSV/JSON outputs document all transitions.

Source: [Chicago Booth Dominick's dataset](https://www.chicagobooth.edu/research/kilts/research-data/dominicks).

### Key Assumptions

- `OK == 1` defines movement eligibility.
- Price eligibility additionally requires `PRICE > 0` and `QTY > 0`.
- Blank `SALE` is unflagged/unknown, not confirmed non-promotion.
- No panel value is imputed and a week gap breaks price-comparison eligibility.
- The three approved T-shirt UPCs are excluded only from CORE and downstream through `config/product_exclusions.csv`.
- Decimal `PRICE` is primary; `PRICE_HEX` is retained for validation.

## Data

The following cell performs an intentional full rebuild using `--overwrite`. The command writes to temporary files, validates them, and only then replaces the named generated outputs. Raw-file hashes are checked before and after. The notebook itself uses only the Python standard library; the reusable pipeline owns tabular processing dependencies.

In [ ]:
from pathlib import Path
import csv
import json
import os
import subprocess
import sys

ROOT = Path.cwd()
if ROOT.name == 'notebooks':
    ROOT = ROOT.parent
CLEAN = ROOT / 'data' / 'processed' / 'clean'
REVIEW = ROOT / 'data' / 'processed' / 'review'

single_thread_env = os.environ.copy()
single_thread_env.update({
    'OMP_NUM_THREADS': '1',
    'OPENBLAS_NUM_THREADS': '1',
    'MKL_NUM_THREADS': '1',
    'VECLIB_MAXIMUM_THREADS': '1',
    'NUMEXPR_NUM_THREADS': '1',
    'KMP_USE_SHM': '0',
})
subprocess.run(
    [sys.executable, str(ROOT / 'src' / 'clean_data.py'), '--root', str(ROOT), '--overwrite'],
    check=True,
    env=single_thread_env,
)

In [ ]:
def read_csv_records(path):
    with path.open(newline='', encoding='utf-8') as handle:
        return list(csv.DictReader(handle))

sample_flow = read_csv_records(CLEAN / 'sample_flow.csv')
promotion_counts = read_csv_records(CLEAN / 'promotion_classification_counts.csv')
invalid_price = read_csv_records(CLEAN / 'invalid_price_summary.csv')
panel_gaps = read_csv_records(CLEAN / 'panel_gap_summary.csv')
product_candidates = read_csv_records(REVIEW / 'product_exclusion_candidates.csv')
manifest = json.loads((CLEAN / 'analytical_sample_manifest.json').read_text())

## Results

### 1. Sample flow

In [ ]:
sample_flow

### 2. Promotion and invalid-price classifications

In [ ]:
promotion_counts, invalid_price

### 3. Panel-gap support fields and product review

In [ ]:
panel_gaps, product_candidates

### 4. Validation assertions

In [ ]:
assert all(manifest['validation'].values())
assert manifest['raw_files_modified'] is False
assert manifest['sample_counts']['VALID_MOVEMENT'] == 6_461_297
assert manifest['sample_counts']['VALID_PRICE'] == 4_707_776
assert manifest['sample_counts']['CORE_CEREAL_PRICE_SAMPLE'] == 4_706_611
assert manifest['sample_counts']['CONSECUTIVE_PRICE_COMPARISON_SAMPLE'] == 4_508_667
assert manifest['product_exclusions']['observations_removed_from_core'] == 1_165
assert len(product_candidates) == 3
manifest['validation']

## Takeaways

- All approved Phase 2 rules are encoded as inspectable fields and named sample layers.
- No `OK=0`, non-positive-price, or non-positive-quantity observation enters its prohibited default layer.
- All 4,508,667 comparison-eligible rows use the immediately preceding eligible calendar week for the same UPC/store.
- The three approved non-food UPCs remove 1,165 observations from CORE while remaining present upstream.
- **Scope:** this notebook constructs and validates the analytical sample layers; price-index, rigidity, promotion, and econometric analyses are documented separately.